In [1]:
# Configuracion de cluster Slurm (alineada con run.sh / run_parallel.sh)
SLURM_PARTITION = 'long'
SLURM_CPUS_PER_TASK = 8
SLURM_MEM = '32G'
SLURM_GRES_EXPERIMENT = 'shard:4'
SLURM_GRES_MASSIVE = 'shard:6'
CONDA_ENV = 'RFA2526pt'
PYTORCH_CUDA_ALLOC_CONF = 'expandable_segments:True'

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = PYTORCH_CUDA_ALLOC_CONF

print('Partition:', SLURM_PARTITION)
print('CPUs:', SLURM_CPUS_PER_TASK, '| Mem:', SLURM_MEM)
print('GRES exp:', SLURM_GRES_EXPERIMENT, '| GRES masivo:', SLURM_GRES_MASSIVE)
print('Conda env:', CONDA_ENV)
print('PYTORCH_CUDA_ALLOC_CONF:', os.environ['PYTORCH_CUDA_ALLOC_CONF'])


Partition: long
CPUs: 8 | Mem: 32G
GRES exp: shard:4 | GRES masivo: shard:6
Conda env: RFA2526pt
PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True


# 05 - Experimento Audio VAD

Objetivo tecnico:
1. Extraer audio desde videos.
2. Eliminar silencios con VAD (WebRTCVAD o fallback con energia de Librosa).
3. Extraer embeddings acusticos con Wav2Vec2.
4. Entrenar regresion logistica en tres configuraciones de idioma (ES, EN, ES+EN).
5. Exportar tres JSON de prediccion sobre el conjunto completo disponible.


In [2]:
import json
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import librosa
import soundfile as sf

import torch
from tqdm.auto import tqdm

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from transformers import AutoProcessor, Wav2Vec2Model

try:
    import webrtcvad
    HAS_WEBRTCVAD = True
except Exception:
    HAS_WEBRTCVAD = False

DATA_JSON_PATH = Path('/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset/training/EXIST2026_training.json')
VIDEO_ROOT = DATA_JSON_PATH.parent
PROJECT_ROOT = Path('/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi')
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DELIVERABLES_DIR = PROJECT_ROOT / 'entregables'
AUDIO_DIR = ARTIFACTS_DIR / 'audio_wav'
CACHE_NPZ = ARTIFACTS_DIR / 'audio_vad_w2v_features.npz'

for d in [ARTIFACTS_DIR, DELIVERABLES_DIR, AUDIO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('HAS_WEBRTCVAD:', HAS_WEBRTCVAD)


Device: cuda
HAS_WEBRTCVAD: False


In [3]:
with open(DATA_JSON_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

rows = []
for sid, obj in raw.items():
    rec = {'sample_id': str(sid)}
    rec.update(obj)
    rows.append(rec)

df = pd.DataFrame(rows)

def majority_task3_1(lbls):
    vals = [x for x in lbls if x in {'YES', 'NO'}]
    if not vals:
        return 'UNKNOWN'
    c = Counter(vals)
    if c['YES'] == c['NO']:
        return vals[0]
    return c.most_common(1)[0][0]

df['label'] = df['labels_task3_1'].apply(majority_task3_1)
df['y'] = df['label'].map({'NO': 0, 'YES': 1}).fillna(-1).astype(int)
df['video_path_abs'] = df['path_video'].apply(lambda p: str((VIDEO_ROOT / p).resolve()))

df = df.sort_values('sample_id').reset_index(drop=True)
print(df[['sample_id', 'lang', 'label']].head(2))
print('n=', len(df), '| langs=', df['lang'].value_counts().to_dict())


  sample_id lang label
0    120001   es   YES
1    120002   es   YES
n= 2524 | langs= {'es': 1524, 'en': 1000}


In [4]:
# Decision de diseno:
# - WebRTCVAD produce un recorte mas estricto para voz hablada.
# - Si no esta disponible, se usa un fallback energetico de librosa.effects.split.
def extract_audio_ffmpeg(video_path: str, wav_path: str, sr: int = 16000):
    if Path(wav_path).exists():
        return wav_path
    cmd = [
        'ffmpeg', '-y', '-i', video_path,
        '-vn', '-ac', '1', '-ar', str(sr), wav_path
    ]
    subprocess.run(cmd, check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return wav_path


def vad_webrtc(y: np.ndarray, sr: int, mode: int = 2):
    vad = webrtcvad.Vad(mode)
    if sr != 16000:
        y = librosa.resample(y, orig_sr=sr, target_sr=16000)
        sr = 16000

    y16 = np.clip(y, -1.0, 1.0)
    pcm = (y16 * 32767).astype(np.int16).tobytes()

    frame_ms = 30
    frame_len = int(sr * frame_ms / 1000) * 2
    voiced = []
    for i in range(0, len(pcm) - frame_len + 1, frame_len):
        frame = pcm[i:i + frame_len]
        if vad.is_speech(frame, sample_rate=sr):
            voiced.append(frame)

    if not voiced:
        return y

    out = np.frombuffer(b''.join(voiced), dtype=np.int16).astype(np.float32) / 32767.0
    return out


def vad_librosa(y: np.ndarray, sr: int, top_db: int = 30):
    intervals = librosa.effects.split(y, top_db=top_db)
    if len(intervals) == 0:
        return y
    return np.concatenate([y[s:e] for s, e in intervals])


def clean_audio_with_vad(wav_path: str):
    y, sr = librosa.load(wav_path, sr=16000, mono=True)
    if HAS_WEBRTCVAD:
        yc = vad_webrtc(y, sr)
    else:
        yc = vad_librosa(y, sr)

    if yc is None or len(yc) < 3200:
        yc = y
    return yc.astype(np.float32), 16000


In [5]:
# Embeddings acusticos Wav2Vec2 con promedio temporal del ultimo hidden state.
MODEL_NAME = 'facebook/wav2vec2-base-960h'
processor = AutoProcessor.from_pretrained(MODEL_NAME)
audio_model = Wav2Vec2Model.from_pretrained(MODEL_NAME).to(DEVICE)
audio_model.eval()

if CACHE_NPZ.exists():
    cache = np.load(CACHE_NPZ, allow_pickle=True)
    X_audio = cache['X_audio']
    sid_cache = cache['sample_ids'].astype(str)
    assert list(sid_cache) == df['sample_id'].astype(str).tolist()
    print('Loaded cache:', CACHE_NPZ, '| shape=', X_audio.shape)
else:
    embs = []
    for row in tqdm(df.itertuples(index=False), total=len(df), desc='Audio VAD + Wav2Vec2'):
        wav_path = AUDIO_DIR / f"{row.sample_id}.wav"
        extract_audio_ffmpeg(row.video_path_abs, str(wav_path))

        y_clean, sr = clean_audio_with_vad(str(wav_path))
        inputs = processor(y_clean, sampling_rate=sr, return_tensors='pt', padding=True)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            hid = audio_model(**inputs).last_hidden_state
            emb = hid.mean(dim=1).squeeze(0).cpu().numpy()
        embs.append(emb)

    X_audio = np.vstack(embs).astype(np.float32)
    np.savez_compressed(CACHE_NPZ, X_audio=X_audio, sample_ids=df['sample_id'].astype(str).values)
    print('Saved cache:', CACHE_NPZ, '| shape=', X_audio.shape)


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Audio VAD + Wav2Vec2:   0%|          | 0/2524 [00:00<?, ?it/s]

ImportError: Numba needs NumPy 2.3 or less. Got NumPy 2.4.

In [ ]:
def train_and_export_three_lang_models(X, y, langs, sample_ids, exp_tag):
    configs = {
        'ES': ['es'],
        'EN': ['en'],
        'ES_EN': ['es', 'en'],
    }

    for cfg_name, cfg_langs in configs.items():
        m = np.isin(langs, cfg_langs) & (y >= 0)
        Xtr, ytr = X[m], y[m]

        clf = Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(max_iter=2500, class_weight='balanced', n_jobs=-1)),
        ])
        clf.fit(Xtr, ytr)

        pred = clf.predict(X)
        pred_labels = np.where(pred == 1, 'YES', 'NO')

        out = [
            {'test_case': 'EXIST2026', 'id': str(sid), 'value': str(lbl)}
            for sid, lbl in zip(sample_ids, pred_labels)
        ]
        out_path = DELIVERABLES_DIR / f'BeingChillingWeWillWin_{exp_tag}_{cfg_name}.json'
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(out, f, ensure_ascii=False)
        print('Saved', out_path, '| rows=', len(out))


train_and_export_three_lang_models(
    X_audio,
    df['y'].to_numpy(np.int64),
    df['lang'].astype(str).to_numpy(),
    df['sample_id'].astype(str).to_numpy(),
    exp_tag='05AudioVAD_W2V2',
)
